In [45]:
from dataclasses import dataclass
from collections import defaultdict

@dataclass
class Rect:
    x1: int
    y1: int
    x2: int
    y2: int

def overlap_1d(xl1, xr1, xl2, xr2):
    return True if max(xl1, xl2) < min(xr1, xr2) else False

def overlap_1d_ok_just(xl1, xr1, xl2, xr2):
    return True if max(xl1, xl2) <= min(xr1, xr2) else False

def overlap_1d_vec_ok_just(vec1: tuple[int, int], vec2: tuple[int, int]):
    return overlap_1d_ok_just(vec1[0], vec1[1], vec2[0], vec2[1])

def overlap_1d_vec(vec1: tuple[int, int], vec2: tuple[int, int]):
    return overlap_1d(vec1[0], vec1[1], vec2[0], vec2[1])

def get_overlap_vec(vec1: tuple[int, int], vec2: tuple[int, int]):
    return (max(vec1[0], vec2[0]), min(vec1[1], vec2[1]))

def concat_1d_vec(vecs: list[tuple[int, int]]):
    serched_vecs = []
    for vec in vecs:
        overlap_vecs = [vec]
        non_overlap_vecs = []
        for seached_vec in serched_vecs:
            if overlap_1d_vec_ok_just(vec, seached_vec):
                overlap_vecs.append(seached_vec)
            else:
                non_overlap_vecs.append(seached_vec)
        concat_overlap_vec = (min([v[0] for v in overlap_vecs]), max([v[1] for v in overlap_vecs]))
        serched_vecs = non_overlap_vecs + [concat_overlap_vec]
    return serched_vecs

def evaluate_1d_partition_costs(pre_vecs: list[tuple[int, int]], next_vecs: list[tuple[int, int]]):
    pre_vecs = concat_1d_vec(pre_vecs)
    next_vecs = concat_1d_vec(next_vecs)

    overlaped_vecs = []
    for pre_vec in pre_vecs:
        for next_vec in next_vecs:
            if overlap_1d_vec(pre_vec, next_vec):
                overlaped_vecs.append(get_overlap_vec(pre_vec, next_vec))
    
    cost = 0
    converged_overlaped_vecs = concat_1d_vec(overlaped_vecs)
    for vec in converged_overlaped_vecs:
        cost += vec[1] - vec[0]
    return cost

def evaluate_partition_rect_cost(pre_rect:list[Rect], next_rect: list[Rect]):
    pre_rect_x_vecs = defaultdict(list)
    pre_rect_y_vecs = defaultdict(list)
    next_rect_x_vecs = defaultdict(list)
    next_rect_y_vecs = defaultdict(list)

    for rect in pre_rect:
        pre_rect_x_vecs[rect.y1].append((rect.x1, rect.x2))
        pre_rect_x_vecs[rect.y2].append((rect.x1, rect.x2))
        pre_rect_y_vecs[rect.x1].append((rect.y1, rect.y2))
        pre_rect_y_vecs[rect.x2].append((rect.y1, rect.y2))        
    for rect in next_rect:
        next_rect_x_vecs[rect.y1].append((rect.x1, rect.x2))
        next_rect_x_vecs[rect.y2].append((rect.x1, rect.x2))
        next_rect_y_vecs[rect.x1].append((rect.y1, rect.y2))
        next_rect_y_vecs[rect.x2].append((rect.y1, rect.y2))
    
    cost = 0
    for key in pre_rect_x_vecs.keys():
        if key not in next_rect_x_vecs:
            continue
        cost += evaluate_1d_partition_costs(pre_rect_x_vecs[key], next_rect_x_vecs[key])
    for key in pre_rect_y_vecs.keys():
        if key not in next_rect_y_vecs:
            continue
        cost += evaluate_1d_partition_costs(pre_rect_y_vecs[key], next_rect_y_vecs[key])
    
    return cost

def evaluate_area_cost(target_rects: list[Rect], target_areas: list[int]):
    cost = 0
    for i in range(len(target_rects)):
        rect_area = abs(target_rects[i].x2 - target_rects[i].x1) * abs(target_rects[i].y2 - target_rects[i].y1)
        if rect_area < target_areas[i]:
            cost +=  target_areas[i] - rect_area
    return cost

def evaluate_cost(pre_rect: list[Rect] | None, next_rect: list[Rect], target_areas: list[int]):
    if pre_rect is None:
        return evaluate_area_cost(next_rect, target_areas)
    return evaluate_partition_rect_cost(pre_rect, next_rect) + evaluate_area_cost(next_rect, target_areas)

def evaluate_all_cost(ans: list[list[int]], target_areas: list[list[int]], days: int):
    
    all_rects = []
    for day in range(days):
        rects = []
        for vec in ans[day]:
            rects.append(Rect(vec[0], vec[1], vec[2], vec[3]))
        all_rects.append(rects)
    
    cost = 0
    for day in range(days):
        if day == 0:
            cost += evaluate_cost(None, all_rects[day], target_areas[day])
        else:
            cost += evaluate_cost(all_rects[day-1], all_rects[day], target_areas[day])
    
    return cost

In [46]:
vecs = [(1, 3), (2, 4), (5, 7), (6, 8), (4.5, 9), (10, 11), (10.5, 12), (12, 15)]

converged_vecs1 = concat_1d_vec(vecs)
converged_vecs1

[(1, 4), (4.5, 9), (10, 15)]

In [47]:
vecs = [(6.5, 7), (10.5, 13), (9, 11), (20, 25), (22, 24)]

converged_vecs2 = concat_1d_vec(vecs)
converged_vecs2

[(6.5, 7), (9, 13), (20, 25)]

In [48]:
evaluate_all_cost([[1, 3, 2, 4, 5, 7, 6, 8, 4.5, 9, 10, 11, 10.5, 12, 12, 15], [6.5, 7, 10.5, 13, 9, 11, 20, 25, 22, 24]], [[2, 2, 2, 2, 2], [2, 2, 2, 2, 2]], 2)

TypeError: 'int' object is not subscriptable